# Note 04: Scaling CUDA Kernels with Grid-Stride Loops

In the previous vector addition example, each GPU thread handled exactly one element.

The pattern looked like this:

```cpp
int i = blockIdx.x * blockDim.x + threadIdx.x;

if (i < N) {
    c[i] = a[i] + b[i];
}
```

This works well when we launch enough threads to cover the entire input.

For example, if:

```text
N = 1000
```

and:

```text
threads_per_block = 256
```

we could launch four blocks:

```text
4 blocks × 256 threads = 1024 threads
```

That gives us enough threads to cover all 1000 elements.

But CUDA kernels do not always need to use the rule:

```text
one thread = one element
```

A thread can process more than one element.

A common CUDA pattern for doing this is called a **grid-stride loop**.

---

# Why Use a Grid-Stride Loop?

Suppose we have:

```text
N = 1000 elements
```

but deliberately launch only:

```text
2 blocks
256 threads per block
```

That gives:

```text
2 × 256 = 512 threads
```

We have:

```text
1000 elements
```

but only:

```text
512 threads
```

If every thread processed only one element, then elements:

```text
512 → 999
```

would never be processed.

Instead, we allow every thread to process another element after finishing its first one.

The thread jumps forward through the array by a fixed distance.

That distance is called the **stride**.

---

# The Grid-Stride Kernel

```cpp
__global__ void stride_vec_add(
    const int* a,
    const int* b,
    int* c,
    const int N
) {

    int i =
        blockIdx.x * blockDim.x
        + threadIdx.x;

    int stride =
        blockDim.x * gridDim.x;

    for (; i < N; i += stride) {
        c[i] = a[i] + b[i];
    }
}
```

There are three important parts here.

---

# 1. Find the Thread's Starting Position

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;
```

This is the normal global thread index.

For:

```text
2 blocks
256 threads per block
```

the global thread IDs are:

```text
Block 0:

0 → 255

Block 1:

256 → 511
```

So every thread gets a different starting position.

For example:

```text
Thread 0 starts at element 0

Thread 1 starts at element 1

Thread 388 starts at element 388

Thread 511 starts at element 511
```

---

# 2. Calculate the Stride

```cpp
int stride =
    blockDim.x * gridDim.x;
```

Here:

```text
blockDim.x = number of threads in one block

gridDim.x = number of blocks in the grid
```

Our launch is:

```cpp
stride_vec_add<<<2, 256>>>(...);
```

Therefore:

```text
blockDim.x = 256

gridDim.x = 2
```

So:

```text
stride = 256 × 2

stride = 512
```

In this example, the stride is the total number of threads in the grid.

---

# What Does "Stride" Mean?

Stride simply means:

> how far we move forward each time.

If:

```text
start = 2
stride = 8
```

then the positions would be:

```text
2
10
18
26
34
...
```

We keep adding `8`.

In our CUDA kernel:

```text
stride = 512
```

so a thread jumps forward by:

```text
512 elements
```

after processing an element.

---

# 3. The Grid-Stride Loop

```cpp
for (; i < N; i += stride) {
    c[i] = a[i] + b[i];
}
```

Normally a C++ loop looks like:

```cpp
for (int i = 0; i < N; i++)
```

It has three parts:

```text
initialization
condition
update
```

But here we already calculated `i` before the loop:

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;
```

So the initialization section inside the `for` loop is empty:

```cpp
for (; i < N; i += stride)
```

The loop means:

```text
while i is inside the array

    process element i

    then move forward by stride
```

---

# Example: Thread 0

Thread 0 begins with:

```text
i = 0
```

The stride is:

```text
512
```

So Thread 0 processes:

```text
0
```

Then:

```text
0 + 512 = 512
```

So it also processes:

```text
512
```

Then:

```text
512 + 512 = 1024
```

But:

```text
1024 >= 1000
```

so the loop stops.

Thread 0 therefore handles:

```text
0
512
```

---

# Example: Thread 1

Thread 1 begins at:

```text
1
```

Then:

```text
1 + 512 = 513
```

So Thread 1 handles:

```text
1
513
```

---

# Example: Thread 388

Thread 388 begins at:

```text
388
```

Then:

```text
388 + 512 = 900
```

So Thread 388 handles:

```text
388
900
```

Then:

```text
900 + 512 = 1412
```

Since:

```text
1412 >= 1000
```

the loop stops.

This is why element `900` gets computed even though only 512 threads were launched.

---

# How the Threads Cover the Array

With:

```text
N = 1000

Total threads = 512

Stride = 512
```

the first pass covers:

```text
0 → 511
```

The second pass covers:

```text
512 → 999
```

Conceptually:

```text
First pass

Thread 0   → element 0
Thread 1   → element 1
Thread 2   → element 2
...
Thread 511 → element 511


Second pass

Thread 0   → element 512
Thread 1   → element 513
Thread 2   → element 514
...
Thread 487 → element 999
```

Threads whose next index is outside the array simply stop.

---

# Using `std::vector`

For the larger example, the host arrays were created using:

```cpp
std::vector<int> h_a(N);
std::vector<int> h_b(N);
std::vector<int> h_c(N);
```

This creates one vector containing `N` integers.

For example:

```cpp
std::vector<int> h_a(N);
```

with:

```text
N = 1000
```

creates space for:

```text
1000 integers
```

on the CPU.

We can access the elements normally:

```cpp
h_a[i] = i;
```

---

# Why `.data()` Is Used

`cudaMemcpy` needs a pointer to the beginning of the host data.

A `std::vector` is a C++ object, so we use:

```cpp
h_a.data()
```

to get a pointer to its first element.

Conceptually:

```text
h_a

C++ vector object

        |
        | .data()
        v

[0][1][2][3][4]...
 ^
 pointer to first integer
```

So:

```cpp
cudaMemcpy(
    d_a,
    h_a.data(),
    bytes,
    cudaMemcpyHostToDevice
);
```

means:

> Copy `bytes` starting from the first integer inside `h_a` into GPU memory pointed to by `d_a`.

With a raw C-style array, the array name itself already behaves like a pointer in this situation.

With `std::vector`, `.data()` gives us that pointer explicitly.

---

# Our Launch Configuration

We deliberately used:

```cpp
const int threads_per_block = 256;
const int blocks = 2;
```

So:

```text
Total threads

= blocks × threads_per_block

= 2 × 256

= 512
```

The input contains:

```text
1000 elements
```

This is intentional.

The goal is to demonstrate that a grid-stride loop allows a smaller grid to process a larger input.

---

# Result

The program printed:

```text
N: 1000
Blocks: 2
Threads per block: 256
Total threads: 512

0 + 0 = 0
1 + 10 = 11
2 + 20 = 22
3 + 30 = 33
4 + 40 = 44
5 + 50 = 55
6 + 60 = 66
7 + 70 = 77
8 + 80 = 88
9 + 90 = 99
```

More importantly:

```text
Check element 900:

900 + 9000 = 9900
```

Element `900` is well beyond the first 512 elements.

It was still processed because the thread responsible for it returned to the loop after processing its first element.

In this case:

```text
Thread 388

first element  → 388

second element → 388 + 512
               → 900
```

---

# One-Thread-Per-Element vs Grid-Stride Loop

The original style:

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;

if (i < N) {
    c[i] = a[i] + b[i];
}
```

usually gives:

```text
one thread
    ↓
one element
```

A grid-stride loop gives:

```text
one thread
    ↓
element i
    ↓
element i + stride
    ↓
element i + 2 × stride
    ↓
...
```

until the index moves outside the array.

---

# Why This Pattern Is Useful

Grid-stride loops separate:

```text
number of data elements
```

from:

```text
number of threads launched
```

The input might contain:

```text
1,000
1,000,000
100,000,000
```

elements.

The kernel does not necessarily need one unique thread for every element.

Instead, threads can repeatedly process elements separated by the grid stride.

This makes the kernel more flexible and allows the launch configuration to be chosen independently from the exact input size.

Grid-stride loops are a common CUDA programming pattern for scalable 1D kernels.

---

# Final Mental Model

A grid-stride loop can be understood as:

```text
Give every thread a starting index

        ↓

Process that element

        ↓

Jump forward by the total grid size

        ↓

Process the next element

        ↓

Jump again

        ↓

Repeat until the index is outside the input
```

In code:

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;

int stride =
    blockDim.x * gridDim.x;

for (; i < N; i += stride) {
    c[i] = a[i] + b[i];
}
```

The important idea is:

```text
global thread ID
=
where the thread starts

stride
=
how far the thread jumps

i < N
=
when the thread must stop
```

This allows a fixed number of GPU threads to process an input that may contain many more elements than there are threads.

In [ ]:
%%writefile grid_stride_vec_add.cu

#implementation

#include <iostream>
#include <vector>
#include <cuda_runtime.h>

__global__ void stride_vec_add(const int* a,const int*b,int *c, const int N ){

  int i = blockIdx.x * blockDim.x + threadIdx.x;
  int stride = blockDim.x * gridDim.x;

  for (; i<N; i+=stride){
    c[i]=a[i]+b[i];
  }

}


int main(){

  const int N = 1000;
  const int bytes = N * sizeof(int);

  std::vector<int> h_a(N);
  std::vector<int> h_b(N);
  std::vector<int> h_c(N);

  int* d_a;
  int* d_b;
  int* d_c;

  for (int i = 0; i < N; i++) {
        h_a[i] = i;
        h_b[i] = i * 10;
    }

    cudaMalloc(&d_a,bytes);
    cudaMalloc(&d_b,bytes);
    cudaMalloc(&d_c,bytes);

    cudaMemcpy(d_a,h_a.data(),bytes,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,h_b.data(),bytes,cudaMemcpyHostToDevice);

    const int threads_per_block= 256;
    const int blocks= 2;

    std::cout << "N: " << N << "\n";
    std::cout << "Blocks: " << blocks << "\n";
    std::cout << "Threads per block: " << threads_per_block << "\n";
    std::cout << "Total threads: "
              << blocks * threads_per_block
              << "\n";

    stride_vec_add<<<blocks,threads_per_block>>>(d_a,d_b,d_c,N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_c.data(),d_c,bytes,cudaMemcpyDeviceToHost);

    // Verify first 10 results
    for (int i = 0; i < 10; i++) {
        std::cout
            << h_a[i]
            << " + "
            << h_b[i]
            << " = "
            << h_c[i]
            << "\n";
    }

    // Verify a value beyond the first 512 threads
    std::cout << "\nCheck element 900:\n";

    std::cout
        << h_a[900]
        << " + "
        << h_b[900]
        << " = "
        << h_c[900]
        << "\n";


cudaFree(d_a);
cudaFree(d_b);
cudaFree(d_c);


return 0;
}

Overwriting grid_stride_vec_add.cu


In [ ]:
!nvcc grid_stride_vec_add.cu -o grid_stride_vec_add
!./grid_stride_vec_add

N: 1000
Blocks: 2
Threads per block: 256
Total threads: 512
0 + 0 = 0
1 + 10 = 11
2 + 20 = 22
3 + 30 = 33
4 + 40 = 44
5 + 50 = 55
6 + 60 = 66
7 + 70 = 77
8 + 80 = 88
9 + 90 = 99

Check element 900:
900 + 9000 = 9900


# Choosing a Grid Size Independently of the Input

A grid-stride loop gives us an important new freedom:

```text
number of elements
```

does not have to directly determine:

```text
number of blocks
```

With the simple one-thread-per-element approach, we normally calculate:

```cpp
const int blocks =
    (N + threads_per_block - 1)
    / threads_per_block;
```

This creates enough threads to cover every element once.

That is completely valid.

But with a grid-stride loop, the kernel can still process the entire input even if the number of launched threads is smaller than `N`.

This means the grid size can sometimes be chosen based on the GPU or workload instead of being calculated directly from the input size.

---

## Querying the GPU

CUDA provides runtime functions that allow the host program to ask for information about the current GPU.

For example:

```cpp
int numSMs;

cudaDeviceGetAttribute(
    &numSMs,
    cudaDevAttrMultiProcessorCount,
    0
);
```

This asks CUDA:

> How many Streaming Multiprocessors does device 0 have?

The result is stored inside:

```cpp
numSMs
```

For example, if the GPU contains:

```text
40 SMs
```

then after the call:

```text
numSMs = 40
```

---

## Why `&numSMs`?

This uses the same pointer idea we already saw with:

```cpp
cudaMalloc(&d_a, bytes);
```

CUDA needs to write a result into the variable.

So we give CUDA the address of:

```cpp
numSMs
```

using:

```cpp
&numSMs
```

Conceptually:

```text
CUDA obtains device information

        ↓

finds the SM count

        ↓

writes the result into numSMs
```

---

## Choosing Blocks Using the GPU

We could then write something like:

```cpp
int blocks = 32 * numSMs;
```

If:

```text
numSMs = 40
```

then:

```text
blocks = 32 × 40

blocks = 1280
```

and launch:

```cpp
kernel<<<blocks, 256>>>(...);
```

The grid-stride loop would then keep processing elements until the entire input is complete.

The important idea is not the number `32`.

That is not a universal formula that should be memorized.

The important idea is:

```text
with a grid-stride loop,

grid size
does not need to equal
input size / threads per block
```

The grid can be chosen separately.

---

## Why Not Launch Very Few Blocks?

A grid-stride loop makes a kernel correct even when we launch fewer threads than elements.

But that does not mean we should always launch a tiny number of blocks.

For example:

```cpp
const int blocks = 2;
```

was useful for demonstrating how grid-stride loops work.

It proved that:

```text
512 threads
```

could process:

```text
1000 elements
```

But on a real GPU, launching only two blocks may leave a large amount of hardware unused.

So there are two separate questions:

```text
Will this kernel process every element correctly?
```

and:

```text
Will this launch configuration use the GPU efficiently?
```

Grid-stride loops help with the first question.

Choosing an efficient launch configuration is a performance problem that we will study later when we learn about:

```text
SMs
warps
occupancy
latency hiding
GPU architecture
```

---

# Final Mental Model

There are now two common ways to write a simple 1D CUDA kernel.

## One Thread Per Element

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;

if (i < N) {
    c[i] = a[i] + b[i];
}
```

The launch usually creates enough threads to cover the whole input.

```text
N
↓
calculate enough blocks
↓
one thread handles one element
```

---

## Grid-Stride Loop

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;

int stride =
    blockDim.x * gridDim.x;

for (; i < N; i += stride) {
    c[i] = a[i] + b[i];
}
```

A thread may process multiple elements.

```text
thread gets starting index
        ↓
process element
        ↓
jump forward by stride
        ↓
process another element
        ↓
repeat until i >= N
```

The three important values are:

```text
global thread ID
=
where the thread starts

stride
=
how far the thread jumps

i < N
=
when the thread stops
```

This makes the kernel independent of having exactly one thread for every data element.

---

# Note 04 Complete

In this note, we learned how CUDA kernels scale beyond small examples.

We covered:

- scaling kernels to arbitrary input sizes
- calculating enough blocks using ceiling division
- why bounds checks are necessary
- how `gridDim.x` represents the number of blocks in the grid
- how stride is calculated from the total number of threads
- grid-stride loops
- how one thread can process multiple elements
- why input size and grid size do not need to be directly tied together
- why a correct grid size and an efficient grid size are separate ideas
- how CUDA can query basic GPU properties such as SM count

At this point, the 1D scaling model is complete.

The next note moves from 1D execution into:

# Note 05: 2D and 3D CUDA Indexing

This will introduce:

```text
dim3
threadIdx.y
blockIdx.y
blockDim.y
gridDim.y
```

and show how CUDA threads map naturally to:

```text
matrices
images
2D data
```